# Energy per Atom of Gold Nanoclusters

> **A. H. Larsen, J. Kleis, K. S. Thygesen, J. K. Nørskov, and K. W. Jacobsen**,
> "Electronic shell structure and chemisorption on gold nanoparticles",
> *Phys. Rev. B 84, 245429 (2011)*,
> [DOI: 10.1103/PhysRevB.84.245429](https://doi.org/10.1103/PhysRevB.84.245429)

Relaxes the cuboctahedral and icosahedral gold clusters created in the [structure notebook](nanocluster_gold.ipynb),
computes their energy per atom relative to bulk gold, and compares it with Fig. 4 of the manuscript.

<h2 style="color:green">Usage</h2>

1. Create the clusters in the [structure notebook](nanocluster_gold.ipynb), which saves them to the `uploads` folder as `Au<N> <Shape>`, e.g. `Au55 Icosahedron`.
1. Set the shapes, sizes and parameters in cells 1.2-1.4 (or use the defaults).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the cluster names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the clusters by name and bulk gold from Standata, print provenance, and save them to the platform.
1. Configure the DFT model.
1. Configure compute: get the list of computing clusters and create a compute configuration.
1. Run one total energy job on bulk gold.
1. Relax each cluster at fixed cell.
1. Retrieve the total energies and compute the energy per atom relative to bulk.
1. Compare with Larsen et al. (2011), Fig. 4.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2. Material names

In [ ]:
# Names saved by nanocluster_gold.ipynb.
ELEMENT = "Au"
SHAPES = ["icosahedron"]  # , "cuboctahedron"
CLUSTER_SIZES = [55]  # list(range(30, 61))
CLUSTER_NAMES = {(shape, size): f"{ELEMENT}{size} {shape.capitalize()}" for shape in SHAPES for size in CLUSTER_SIZES}
BULK_NAME = "Gold"
COMPARISON_SIZES = [34, 40, 55, 58, 60]  # sizes printed beside the manuscript's values

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = None
QUEUE_NAME = QueueName.OR
PPN = 16  # OR allows 16 cores per job
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
# manuscript: RPBE with a localized (DZP) basis set in GPAW.
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry"

BULK_KGRID = [16, 16, 16]
CLUSTER_KGRID = [1, 1, 1]  # Γ point, the clusters are isolated
RELAXATION_SETTINGS = {"forc_conv_thr": 2.9e-3, "nstep": 100}  # Ry/Bohr, the manuscript's 0.075 eV/Å

SCF_UNIT = "pw_scf"
RELAX_UNIT = "pw_relax"

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load the clusters from the uploads folder or the platform, and bulk gold from Standata

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.core.entity.material.api import load_material

materials_by_name = {BULK_NAME: Material.create(Materials.get_by_name_first_match(BULK_NAME))}
materials_by_name.update({name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in CLUSTER_NAMES.values()})
for material in materials_by_name.values():
    a, b, c = material.lattice.a, material.lattice.b, material.lattice.c
    print(f"{material.name}: {material.basis.number_of_atoms} atoms, cell {a:.3f} x {b:.3f} x {c:.3f} Å")

### 3.2. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {}
for name, material in materials_by_name.items():
    material.basis.set_labels_from_list([])
    saved_materials[name] = get_or_create_material(client, material, ACCOUNT_ID)

## 4. Configure the DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

application_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
application = Application(**application_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {application.name}, model: {MODEL_TAG}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[cluster['hostname'] for cluster in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((candidate for candidate in clusters if CLUSTER_NAME in candidate["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. "
                         f"Available: {[candidate['hostname'] for candidate in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Total energy of bulk gold

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material


def find_or_create_job(name, workflow_config, workflow_label, unit_name, kgrid, qe_parameters):
    """Returns this material's job id for the workflow, creating the job if none exists, and whether it did."""
    workflow = Workflow.create(workflow_config)
    workflow.name = f"{workflow_label} {name} {MODEL_TAG}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)
    apply_scf_kgrid(workflow, kgrid, material=materials_by_name[name], unit_name=unit_name)
    patch_workflow_qe_input(workflow, qe_parameters, [unit_name])
    job = find_job_for_material(client, saved_materials[name]["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=kgrid, unit_name=unit_name)
    if job is not None:
        print(f"♻️  {name}: reusing existing {workflow_label} job {job['_id']}")
        return job["_id"], False
    job = create_job(
        api_client=client, materials=[saved_materials[name]], workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
    )
    print(f"✅ {name}: created {workflow_label} job {job['_id']}")
    return job["_id"], True

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

total_energy_workflow_config = WorkflowStandata.filter_by_application(application.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)
bulk_job_id, is_new = find_or_create_job(BULK_NAME, total_energy_workflow_config, "Total Energy", SCF_UNIT,
                                         BULK_KGRID, {})
if is_new:
    submit_jobs(client.jobs, [bulk_job_id])
await wait_for_jobs_to_finish_async(client.jobs, [bulk_job_id], poll_interval=POLL_INTERVAL)

## 7. Relax the clusters

Every atom of each cluster relaxes at fixed cell; the cluster's energy is the final energy of the relaxation.

In [ ]:
relax_workflow_config = WorkflowStandata.filter_by_application(application.name).get_by_name_first_match(
    RELAX_WORKFLOW_SEARCH_TERM
)
relax_job_ids = {}
new_relax_job_ids = []
for name in CLUSTER_NAMES.values():
    relax_job_ids[name], is_new = find_or_create_job(name, relax_workflow_config, "Fixed-cell Relaxation",
                                                     RELAX_UNIT, CLUSTER_KGRID, {"control": RELAXATION_SETTINGS})
    if is_new:
        new_relax_job_ids.append(relax_job_ids[name])

if new_relax_job_ids:
    submit_jobs(client.jobs, new_relax_job_ids)
    print(f"✅ Submitted {len(new_relax_job_ids)} relaxation job(s).")
await wait_for_jobs_to_finish_async(client.jobs, list(relax_job_ids.values()), poll_interval=POLL_INTERVAL)

## 8. Retrieve the results
### 8.1. Total energies

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job

bulk_energy_per_atom = (get_properties_for_job(client, bulk_job_id, "total_energy")[0]["value"]
                        / materials_by_name[BULK_NAME].basis.number_of_atoms)
print(f"{BULK_NAME}: {bulk_energy_per_atom:.4f} eV/atom")
cluster_energy = {}
for name, job_id in relax_job_ids.items():
    cluster_energy[name] = get_properties_for_job(client, job_id, "total_energy")[0]["value"]
    total_force = get_properties_for_job(client, job_id, "total_force")[0]
    print(f"{name}: {cluster_energy[name]:.4f} eV, total force (norm over all atoms) {total_force['value']:.4f} {total_force['units']}")

### 8.2. Energy per atom relative to bulk

As in the manuscript's Fig. 4, the energy of an $N$-atom cluster is measured from $N$ atoms of bulk gold, per atom:

$$\Delta E(N) = \frac{E(\mathrm{Au}_N) - N E_{bulk}}{N},$$

with $E_{bulk}$ the total energy per atom of bulk fcc gold.

In [ ]:
energy_per_atom = {(shape, size): (cluster_energy[name] - size * bulk_energy_per_atom) / size
                   for (shape, size), name in CLUSTER_NAMES.items()}

## 9. Comparison with Larsen et al. (2011)

Fig. 4 of the manuscript, read from the figure: the lowest and highest energy per atom over the four random seeds at
each size, `None` where no point lies in the plotted range (above 0.75 eV/atom). Manuscript: RPBE, localized (DZP)
basis set; ours: PBE, plane waves.

In [ ]:
# eV/atom, (lowest, highest) over the four seeds
LARSEN_2011_FIG4 = {
    "cuboctahedron": {
        30: None, 31: (0.750, 0.750), 32: (0.730, 0.748), 33: (0.715, 0.745), 34: (0.683, 0.720),
        35: (0.683, 0.723), 36: (0.685, 0.728), 37: (0.677, 0.739), 38: (0.689, 0.732), 39: (0.680, 0.731),
        40: (0.683, 0.715), 41: (0.663, 0.730), 42: (0.664, 0.729), 43: (0.697, 0.726), 44: (0.669, 0.708),
        45: (0.657, 0.705), 46: (0.673, 0.726), 47: (0.660, 0.715), 48: (0.657, 0.706), 49: (0.667, 0.697),
        50: (0.635, 0.686), 51: (0.650, 0.674), 52: (0.642, 0.660), 53: (0.644, 0.646), 54: (0.631, 0.633),
        55: (0.615, 0.616), 56: (0.610, 0.610), 57: (0.596, 0.597), 58: (0.559, 0.578), 59: (0.583, 0.583),
        60: (0.584, 0.585),
    },
    "icosahedron": {
        30: (0.740, 0.748), 31: (0.736, 0.741), 32: (0.708, 0.734), 33: (0.696, 0.734), 34: (0.666, 0.703),
        35: (0.669, 0.697), 36: (0.661, 0.695), 37: (0.693, 0.713), 38: (0.689, 0.710), 39: (0.686, 0.710),
        40: (0.690, 0.706), 41: (0.685, 0.694), 42: (0.652, 0.690), 43: (0.676, 0.681), 44: (0.670, 0.680),
        45: (0.664, 0.675), 46: (0.646, 0.664), 47: (0.649, 0.651), 48: (0.640, 0.653), 49: (0.636, 0.641),
        50: (0.627, 0.634), 51: (0.625, 0.635), 52: (0.622, 0.624), 53: (0.605, 0.626), 54: (0.610, 0.610),
        55: (0.602, 0.602), 56: (0.580, 0.585), 57: (0.576, 0.576), 58: (0.556, 0.558), 59: (0.584, 0.584),
        60: (0.554, 0.555),
    },
}

In [ ]:
import matplotlib.pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

figure, axes = plt.subplots()
for index, (shape, ranges) in enumerate(LARSEN_2011_FIG4.items()):
    sizes = [size for size, seeds in ranges.items() if seeds]
    axes.fill_between(sizes, [ranges[size][0] for size in sizes], [ranges[size][1] for size in sizes],
                      color=f"C{index}", alpha=0.3, label=f"{shape.capitalize()}, Larsen 2011")
    if shape in SHAPES:
        ours = {size: value for (ours_shape, size), value in energy_per_atom.items() if ours_shape == shape}
        axes.plot(list(ours), list(ours.values()), "o", color=f"C{index}", label=f"{shape.capitalize()}, ours")
axes.set_xlabel("Number of atoms N")
axes.set_ylabel("ΔE(N) (eV/atom)")
axes.set_title("Energy per atom of Au clusters relative to bulk gold")
axes.legend()
display_matplotlib_figure(figure)

In [ ]:
for (shape, size), value in energy_per_atom.items():
    if size in COMPARISON_SIZES:
        lowest, highest = LARSEN_2011_FIG4[shape][size]
        manuscript = f"{lowest:.3f}" if lowest == highest else f"{lowest:.3f}–{highest:.3f}"
        print(f"ΔE, {CLUSTER_NAMES[(shape, size)]}: Larsen 2011 {manuscript} eV/atom | ours {value:.3f} eV/atom")

## References

[1] A. H. Larsen, J. Kleis, K. S. Thygesen, J. K. Nørskov and K. W. Jacobsen, "Electronic shell structure and
chemisorption on gold nanoparticles", Phys. Rev. B 84, 245429 (2011). https://doi.org/10.1103/PhysRevB.84.245429